**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 22 - SQL y manejo de tablas**

## Complemento

Este notebook se complementa con la presentación: **DATA.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

Todo lo que vimos hasta ahora arrancaba con un archivo. Pero en una organización de verdad los datos
**no viven en archivos sueltos**: viven en una **base de datos**. Y a las bases de datos se les habla
en un idioma que tiene 50 años y no piensa jubilarse: **SQL**.

| Parte | Tema | Cláusula |
|---|---|---|
| **A** | Por qué SQL y no un Excel | — |
| **B** | Armar la base y mirarla | `sqlite3` |
| **C** | Traer datos y filtrar | `SELECT` · `WHERE` · `ORDER BY` |
| **D** | Resumir | `GROUP BY` · `HAVING` |
| **E** | **Cruzar tablas** | `JOIN` |
| **F** | Preguntas dentro de preguntas | Subconsultas |
| **G** | El diccionario SQL ↔ pandas | — |

> **Por qué te sirve:** "¿sabés SQL?" es literalmente la primera pregunta en una entrevista de análisis
> de datos, control de gestión o auditoría. Es la habilidad técnica más pedida y la más fácil de aprender
> de todas las que vimos.

> ✅ **No hay que instalar nada.** Usamos `sqlite3`, que ya viene adentro de Python.

In [ ]:
import sqlite3                     # sqlite3: motor de base de datos incluido en Python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"

---
# 🤔 Parte A — ¿Por qué SQL si ya sé pandas?

Las dos herramientas hacen lo mismo. La diferencia está en **dónde viven los datos**.

| | pandas | SQL |
|---|---|---|
| Dónde están los datos | En la memoria de tu computadora | En un servidor, ordenados |
| Cuánto aguanta | Lo que entre en tu RAM | Millones de filas sin despeinarse |
| Quién más lo usa | Analistas | Analistas, sistemas, la empresa entera |
| Varias personas a la vez | No | Sí |

**En criollo:** pandas es tu escritorio, SQL es el archivo central de la empresa. Vas al archivo central,
pedís *solo lo que necesitás*, y eso lo traés a tu escritorio para trabajarlo.

### Un poco de vocabulario

| Término | En Excel sería | En pandas sería |
|---|---|---|
| **Base de datos** | El archivo entero | — |
| **Tabla** | Una hoja | Un DataFrame |
| **Registro** (fila) | Una fila | Una fila |
| **Campo** (columna) | Una columna | Una columna |
| **Clave primaria** | La columna de ID única | El índice |
| **Consulta** (*query*) | Un filtro o tabla dinámica | Una línea de código |

---
# 🗄️ Parte B — Armamos nuestra base de datos

Vamos a construir la base de una distribuidora de tecnología con **tres tablas**:

| Tabla | Qué guarda | Filas |
|---|---|---|
| `ventas` | Cada operación de venta | 200 |
| `productos` | El catálogo: categoría y costo | 6 |
| `vendedores` | El equipo comercial y su sucursal | 5 |

Esta forma de organizar —una tabla de **hechos** rodeada de tablas de **referencia**— es la más común
en cualquier empresa. Se llama *modelo estrella*.

In [ ]:
# 1) Traemos las ventas del repositorio
ventas = pd.read_csv(URL + "ventas.csv")
ventas["Total"] = ventas["Cantidad"] * ventas["Precio_Unitario"]

# 2) Creamos el catálogo de productos
productos = pd.DataFrame({
    "Producto":  ["Notebook", "Monitor", "Impresora", "Teclado", "Mouse", "Auriculares"],
    "Categoria": ["Cómputo", "Cómputo", "Periférico", "Periférico", "Periférico", "Audio"],
    "Costo":     [62000, 31000, 24000, 8500, 4200, 6800],
})

# 3) Y el equipo de ventas
vendedores = pd.DataFrame({
    "Vendedor":  ["Juan", "Lucía", "María", "Pedro", "Sofía"],
    "Sucursal":  ["Centro", "Norte", "Centro", "Sur", "Norte"],
    "Antiguedad": [8, 3, 12, 1, 5],
})

print("ventas:", ventas.shape, "| productos:", productos.shape, "| vendedores:", vendedores.shape)
ventas.head(3)

In [ ]:
# Creamos la base de datos en memoria y volcamos las tres tablas
con = sqlite3.connect(":memory:")     # ":memory:" = base temporal en RAM (se borra al cerrar)

ventas.to_sql("ventas", con, index=False, if_exists="replace")
productos.to_sql("productos", con, index=False, if_exists="replace")
vendedores.to_sql("vendedores", con, index=False, if_exists="replace")

print("Base creada ✅")

> 📌 **`:memory:`** crea una base que vive solo mientras dure la sesión. Si querés un archivo de verdad
> en el disco, se pone el nombre: `sqlite3.connect("empresa.db")`. Todo lo demás es igual.

In [ ]:
# Esta función la vamos a usar en toda la clase: manda una consulta y devuelve un DataFrame
def sql(consulta):
    return pd.read_sql_query(consulta, con)

# Probémosla: ¿qué tablas tiene la base?
sql("SELECT name FROM sqlite_master WHERE type='table'")

---
# 🔍 Parte C — `SELECT`: traer datos

La estructura básica de toda consulta SQL es siempre la misma:

```sql
SELECT   columnas          -- qué quiero
FROM     tabla             -- de dónde
WHERE    condición         -- qué filas
ORDER BY columna           -- cómo ordenado
LIMIT    n                 -- cuántas
```

Se lee casi como una oración en inglés. Empecemos por lo mínimo.

In [ ]:
# El asterisco * significa "todas las columnas"
sql("SELECT * FROM ventas LIMIT 5")

In [ ]:
# Pedimos solo algunas columnas, en el orden que queramos
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    LIMIT 5
""")

> 💡 Las **comillas triples** `""" """` permiten escribir la consulta en varias líneas.
> A partir de acá las usamos siempre: una consulta SQL bien indentada se lee muchísimo mejor.
>
> Por convención, las **palabras de SQL van en mayúscula** (`SELECT`, `FROM`, `WHERE`) y los nombres de
> tablas y columnas en minúscula. No es obligatorio, pero todo el mundo lo hace.

### `WHERE`: filtrar filas

Es el equivalente exacto de `ventas[ventas["Ciudad"] == "Córdoba"]` en pandas.

In [ ]:
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    WHERE Ciudad = 'Córdoba'
    LIMIT 5
""")

| Operador | Qué hace | Ejemplo |
|---|---|---|
| `=` | Igual (¡uno solo, no `==`!) | `WHERE Ciudad = 'Rosario'` |
| `<>` o `!=` | Distinto | `WHERE Producto <> 'Mouse'` |
| `>` `<` `>=` `<=` | Comparación | `WHERE Total > 100000` |
| `AND` / `OR` | Combinar condiciones | `WHERE Total > 100000 AND Ciudad = 'Rosario'` |
| `IN` | Está en una lista | `WHERE Producto IN ('Monitor', 'Notebook')` |
| `BETWEEN` | Entre dos valores | `WHERE Cantidad BETWEEN 3 AND 6` |
| `LIKE` | Coincide con un patrón | `WHERE Producto LIKE 'Note%'` |
| `IS NULL` | Falta el dato | `WHERE Ciudad IS NULL` |

> ⚠️ **Dos trampas clásicas para el que viene de Python:**
> 1. En SQL la igualdad es **`=`**, no `==`.
> 2. El texto va entre **comillas simples**: `'Córdoba'`.

In [ ]:
# Varias condiciones a la vez
sql("""
    SELECT Fecha, Producto, Ciudad, Cantidad, Total
    FROM ventas
    WHERE Total > 300000
      AND Producto IN ('Notebook', 'Monitor')
    ORDER BY Total DESC
    LIMIT 8
""")

`ORDER BY Total DESC` ordena de mayor a menor (`DESC` = *descending*). Sin el `DESC` ordena de menor a mayor.

---
# 📊 Parte D — `GROUP BY`: resumir

Acá SQL empieza a brillar. Las **funciones de agregación** toman muchas filas y devuelven un solo número:

| Función | Qué calcula |
|---|---|
| `COUNT(*)` | Cuántas filas |
| `SUM(col)` | La suma |
| `AVG(col)` | El promedio |
| `MIN(col)` / `MAX(col)` | El mínimo / máximo |
| `ROUND(col, 2)` | Redondear |

In [ ]:
# Sin GROUP BY: un resumen de TODA la tabla
sql("""
    SELECT COUNT(*)          AS operaciones,
           SUM(Total)        AS facturacion,
           ROUND(AVG(Total)) AS ticket_promedio,
           MAX(Total)        AS venta_mas_grande
    FROM ventas
""")

`AS` le pone nombre a la columna del resultado. Sin eso, la columna se llamaría `COUNT(*)`, que es feo
e incómodo de usar después.

In [ ]:
# Con GROUP BY: el mismo resumen, pero abierto por vendedor
sql("""
    SELECT Vendedor,
           COUNT(*)          AS operaciones,
           SUM(Total)        AS facturacion,
           ROUND(AVG(Total)) AS ticket_promedio
    FROM ventas
    GROUP BY Vendedor
    ORDER BY facturacion DESC
""")

**La regla de oro del `GROUP BY`:** toda columna que aparezca en el `SELECT` y **no** esté dentro de una
función de agregación, tiene que estar en el `GROUP BY`. Es el error más común al empezar.

In [ ]:
# Se puede agrupar por más de una columna
sql("""
    SELECT Ciudad,
           Producto,
           COUNT(*)   AS operaciones,
           SUM(Total) AS facturacion
    FROM ventas
    GROUP BY Ciudad, Producto
    ORDER BY facturacion DESC
    LIMIT 10
""")

### `HAVING`: filtrar **después** de agrupar

Esta distinción entra en el parcial, así que prestale atención:

| Cláusula | Cuándo actúa | Filtra |
|---|---|---|
| `WHERE` | **Antes** de agrupar | Filas individuales |
| `HAVING` | **Después** de agrupar | Grupos ya resumidos |

*"Ciudades cuya facturación total supere el millón"* → eso solo se sabe **después** de sumar. Va en `HAVING`.

In [ ]:
sql("""
    SELECT Ciudad,
           COUNT(*)   AS operaciones,
           SUM(Total) AS facturacion
    FROM ventas
    WHERE Cantidad >= 3            -- filtra FILAS antes de agrupar
    GROUP BY Ciudad
    HAVING SUM(Total) > 3000000    -- filtra GRUPOS después de agrupar
    ORDER BY facturacion DESC
""")

---
# 🔗 Parte E — `JOIN`: cruzar tablas

Este es **el corazón de SQL** y la razón por la que las bases de datos se dividen en varias tablas.

La tabla `ventas` sabe que se vendió una "Notebook", pero **no sabe cuánto cuesta producirla**.
Eso está en `productos`. Para calcular la ganancia hay que **pegar** las dos tablas por la columna
que tienen en común: `Producto`.

```
    ventas                    productos
┌───────────┬───────┐     ┌───────────┬──────────┬───────┐
│ Producto  │ Total │     │ Producto  │Categoria │ Costo │
├───────────┼───────┤     ├───────────┼──────────┼───────┤
│ Notebook  │ 614k  │ ←→  │ Notebook  │ Cómputo  │ 62000 │
│ Monitor   │ 138k  │ ←→  │ Monitor   │ Cómputo  │ 31000 │
└───────────┴───────┘     └───────────┴──────────┴───────┘
                  ↑ la columna en común
```

In [ ]:
sql("""
    SELECT v.Fecha,
           v.Producto,
           p.Categoria,
           v.Cantidad,
           v.Total,
           p.Costo * v.Cantidad         AS costo_total,
           v.Total - p.Costo * v.Cantidad AS ganancia
    FROM ventas v
    JOIN productos p  ON v.Producto = p.Producto
    LIMIT 8
""")

**Desarmemos la consulta:**

- `FROM ventas v` → la tabla `ventas`, a la que le ponemos el apodo (*alias*) `v`.
- `JOIN productos p` → le pegamos `productos`, con el alias `p`.
- `ON v.Producto = p.Producto` → **la condición del cruce**: unir las filas donde el producto coincida.
- `v.Total`, `p.Costo` → el alias evita ambigüedad cuando las dos tablas tienen columnas con el mismo nombre.

> ⚠️ **Si te olvidás el `ON`**, SQL cruza *cada fila con cada fila*: 200 × 6 = 1.200 filas de basura.
> Se llama *producto cartesiano* y es el error más caro de SQL.

In [ ]:
# Ahora la pregunta de negocio: ¿qué categoría deja más margen?
sql("""
    SELECT p.Categoria,
           COUNT(*)                                AS operaciones,
           SUM(v.Total)                            AS facturacion,
           SUM(v.Total - p.Costo * v.Cantidad)     AS ganancia,
           ROUND(100.0 * SUM(v.Total - p.Costo * v.Cantidad) / SUM(v.Total), 1) AS margen_pct
    FROM ventas v
    JOIN productos p ON v.Producto = p.Producto
    GROUP BY p.Categoria
    ORDER BY ganancia DESC
""")

> 💡 Fijate el `100.0` en lugar de `100`. En SQLite, dividir dos enteros da un entero
> (`3 / 2 = 1`). Poniendo un decimal forzás la división real. Es una trampa clásica.

In [ ]:
# Se pueden encadenar varios JOIN: ventas + productos + vendedores
sql("""
    SELECT ve.Sucursal,
           p.Categoria,
           COUNT(*)     AS operaciones,
           SUM(v.Total) AS facturacion
    FROM ventas v
    JOIN productos  p  ON v.Producto = p.Producto
    JOIN vendedores ve ON v.Vendedor = ve.Vendedor
    GROUP BY ve.Sucursal, p.Categoria
    ORDER BY ve.Sucursal, facturacion DESC
""")

### Los tipos de `JOIN`

| Tipo | Qué devuelve | Cuándo usarlo |
|---|---|---|
| `INNER JOIN` (o solo `JOIN`) | Solo lo que coincide en **ambas** tablas | Por defecto |
| `LEFT JOIN` | **Todo** lo de la izquierda + lo que matchee de la derecha | Para no perder filas |
| `RIGHT` / `FULL` | Al revés / todo | SQLite no los soporta; otros motores sí |

**Cuándo importa:** si un producto de `ventas` no estuviera en el catálogo, el `INNER JOIN`
lo **borraría en silencio**. El `LEFT JOIN` lo conserva con `NULL` en las columnas del catálogo.

In [ ]:
# Comprobamos que no estemos perdiendo ventas en el cruce
sql("""
    SELECT (SELECT COUNT(*) FROM ventas)                                    AS filas_originales,
           (SELECT COUNT(*) FROM ventas v JOIN productos p
                            ON v.Producto = p.Producto)                     AS filas_tras_join
""")

Dan igual: no perdimos ninguna venta. **Hacé siempre este control después de un `JOIN`.**

---
# 🪆 Parte F — Subconsultas

Una consulta puede ir **adentro** de otra. Sirve para preguntas de dos pasos, del tipo:
*"¿qué ventas superan el promedio general?"* — primero hay que calcular el promedio.

In [ ]:
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    WHERE Total > (SELECT AVG(Total) FROM ventas)    -- la subconsulta calcula el promedio
    ORDER BY Total DESC
    LIMIT 8
""")

In [ ]:
# Una subconsulta puede hacer de tabla temporal
sql("""
    SELECT Sucursal,
           ROUND(AVG(facturacion)) AS facturacion_promedio_por_vendedor
    FROM (
        SELECT ve.Sucursal, v.Vendedor, SUM(v.Total) AS facturacion
        FROM ventas v
        JOIN vendedores ve ON v.Vendedor = ve.Vendedor
        GROUP BY ve.Sucursal, v.Vendedor
    )
    GROUP BY Sucursal
    ORDER BY facturacion_promedio_por_vendedor DESC
""")

In [ ]:
# El resultado de sql() es un DataFrame de pandas → se grafica como cualquier otro
datos = sql("""
    SELECT Ciudad, SUM(Total) AS facturacion
    FROM ventas
    GROUP BY Ciudad
    ORDER BY facturacion DESC
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(datos["Ciudad"], datos["facturacion"], color="#243b5e")
ax.set_title("Facturación por ciudad", loc="left", fontweight="bold")
ax.set_ylabel("Facturación ($)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
# 🔄 Parte G — El diccionario SQL ↔ pandas

Las dos herramientas hacen lo mismo con distinta sintaxis. Guardate esta tabla:

| Objetivo | SQL | pandas |
|---|---|---|
| Ver todo | `SELECT * FROM t` | `df` |
| Algunas columnas | `SELECT a, b FROM t` | `df[["a", "b"]]` |
| Filtrar | `WHERE a > 5` | `df[df["a"] > 5]` |
| Ordenar | `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| Primeras n | `LIMIT 5` | `df.head(5)` |
| Contar | `COUNT(*)` | `len(df)` |
| Agrupar y sumar | `GROUP BY a` + `SUM(b)` | `df.groupby("a")["b"].sum()` |
| Filtrar grupos | `HAVING SUM(b) > 100` | `.groupby("a").filter(...)` |
| Cruzar tablas | `JOIN t2 ON t.k = t2.k` | `df.merge(df2, on="k")` |
| Valores únicos | `SELECT DISTINCT a` | `df["a"].unique()` |

> 🎯 **En la práctica se usan juntas:** SQL para traer del servidor solo lo que necesitás
> (rápido, poca memoria), y pandas para analizar y graficar. Es exactamente lo que hicimos hoy.

---
# 📝 Ejercicios

Trabajá con la base que ya está cargada. Usá la función `sql("...")`.

**Ejercicio 1.** Traé las 10 ventas de mayor monto, mostrando fecha, producto, vendedor y total.

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** ¿Cuántas operaciones y cuánto facturó cada **ciudad**? Ordenado de mayor a menor.

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** ¿Qué productos tuvieron un ticket promedio mayor a $150.000?
Cuidado: ¿va en `WHERE` o en `HAVING`?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** Con un `JOIN` a `vendedores`, calculá la facturación por **sucursal** y la
antigüedad promedio de su equipo.

In [ ]:
# Tu respuesta acá

**Ejercicio 5.** ¿Cuál es el producto **más rentable** en pesos? (necesitás cruzar con `productos`
y restar el costo).

In [ ]:
# Tu respuesta acá

**Ejercicio 6 (integrador).** Resolvé el Ejercicio 2 **otra vez**, pero con pandas en lugar de SQL.
Compará los dos resultados con `.equals()` y decidí cuál te resultó más cómodo escribir.

In [ ]:
# Tu respuesta acá

**Ejercicio 7 (desafío 🥇⚡🤓).** Para cada vendedor, calculá qué porcentaje de **su** facturación
viene de la categoría "Cómputo". Pista: vas a necesitar una subconsulta o un `JOIN` con un `GROUP BY` adentro.

In [ ]:
# Tu respuesta acá

In [ ]:
con.close()   # buena práctica: cerrar la conexión cuando terminás
print("Conexión cerrada ✅")

---
## 🧭 Para llevarse

```sql
SELECT   columnas, FUNCION(columna) AS alias
FROM     tabla  alias
JOIN     otra_tabla alias2  ON  alias.clave = alias2.clave
WHERE    condición sobre filas
GROUP BY columnas_no_agregadas
HAVING   condición sobre grupos
ORDER BY columna DESC
LIMIT    n
```

**Ese bloque es el 90% del SQL que vas a escribir en tu vida laboral.** El orden de las cláusulas
no es negociable: si ponés el `WHERE` después del `GROUP BY`, no anda.

**Los tres errores que más se cometen:**
1. Usar `==` en vez de `=`.
2. Olvidarse el `ON` en un `JOIN` (y quedarse con miles de filas fantasma).
3. Poner en `WHERE` una condición sobre un total, que va en `HAVING`.